# Análisis de datos

## Inicializa Spark

In [0]:
%run "./00_Init_Spark"

## Importa librerías

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

## Lee datos de tabla final

In [0]:
df = spark.read.table("gold.personas")

In [0]:
df.count()

In [0]:
# Años escolaridad - Region
graph1 = df \
    .where(col("escolaridad").isNotNull()) \
    .select(col("region"), col("escolaridad")) \
    .groupBy(col("region")) \
    .agg(avg(col("escolaridad")).alias("avg_escolaridad")) \
    .orderBy(desc(col("avg_escolaridad")))


## Años de escolaridad promedio por Región

In [0]:
sns.set_theme()

f, ax = plt.subplots(figsize=(3, 6))

# Grafico de barras
sns.set_color_codes("pastel")
sns.barplot(x="avg_escolaridad", y="region", data=graph1.toPandas())

# Numero en las barras
ax.bar_label(ax.containers[0], fmt='%.1f', padding=3)

# Leyendas
ax.legend(ncol=2, loc="lower right", frameon=True)
ax.set(xlim=(0, 15), ylabel="Región",
       xlabel="Años de escolaridad promedio")

sns.despine(left=True, bottom=True)

## Distribución de estado civil por región

In [0]:

df_est_civ = df.where(col("p23_est_civil").isNotNull()) \
    .select("region", "p23_est_civil") \
    .groupBy("region", "p23_est_civil") \
    .count() \
    .orderBy("region", "p23_est_civil") \

pd_hist = df_est_civ.toPandas()

In [0]:
g = sns.catplot(
    data=pd_hist,
    x="p23_est_civil",
    y="count",
    hue="p23_est_civil",
    col="region",
    kind="bar",
    col_wrap=4,
    height=8,
    legend=False, 
)

g.set_axis_labels("Estado civil", "Cantidad de personas")
g.set_titles("{col_name}")

for ax in g.axes.flat:
    ax.tick_params(axis='x', rotation=80)
    ax.ticklabel_format(style='plain', axis='y')

    for p in ax.patches:
        height = p.get_height()
        ax.text(
            x=p.get_x() + p.get_width() / 2,
            y=height,
            s=f"{height:.0f}",
            ha="center", va="bottom",
            fontsize=9
        )

plt.tight_layout()
plt.show()